# RAG with LangChain - Basics

**RAG = Retrieval Augmented Generation.** Give the LLM external knowledge instead of relying only on its training data. Reduces hallucination, lets it use fresh/private/domain data.

## The pipeline
```
Load -> Split -> Embed -> Store -> Retrieve -> Generate
```

| Stage | What it does | LangChain piece |
|-------|--------------|-----------------|
| **Load** | Read raw data into `Document` objects | Document Loaders |
| **Split** | Chop big docs into chunks | Text Splitters (transformers) |
| **Embed** | Turn chunks into vectors | Embeddings |
| Store | Save vectors | Vector stores |
| Retrieve | Find similar chunks | Retrievers |
| Generate | LLM answers using chunks | Chat model |

This notebook covers the first 3 stages: **Loaders, Transformers, Embeddings.**

## 0. Setup

Load API key from `.env`. We use Google Gemini (already configured in this project).

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

print('Key loaded:', bool(os.environ.get('GOOGLE_API_KEY')))

Key loaded: True


---
# 1. Document Loaders

A **loader** reads raw data (txt, pdf, web, csv...) and returns a list of `Document` objects.

Every `Document` has two parts:
- `page_content` -> the actual text (a string)
- `metadata` -> a dict (source file, page number, url, etc.)

Loaders live in `langchain_community.document_loaders`. All loaders share the same API: `.load()` returns `List[Document]`.

### 1a. TextLoader - plain `.txt` files

Simplest loader. One file -> one `Document`.

In [12]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('sample.txt', encoding='utf-8')
docs = loader.load()

print('Number of documents:', len(docs))
print('Type:', type(docs[0]))
print('Metadata:', docs[0].metadata)
print('---- content (first 200 chars) ----')
print(docs[0].page_content[:200])

Number of documents: 1
Type: <class 'langchain_core.documents.base.Document'>
Metadata: {'source': 'sample.txt'}
---- content (first 200 chars) ----
RAG stands for Retrieval Augmented Generation.

It combines a retriever (which fetches relevant documents from a knowledge base)
with a generator (an LLM that produces the final answer). Instead of re


### 1b. PyPDFLoader - PDF files

Reads a PDF. Returns **one Document per page**, with the page number in metadata. (`pypdf` is already a dependency. `resume.pdf` is in this folder.)

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader('resume.pdf')
pdf_docs = pdf_loader.load()

print('Pages loaded:', len(pdf_docs))
print('Page 0 metadata:', pdf_docs[0].metadata)
print('---- page 0 content (first 300 chars) ----')
print(pdf_docs[0].page_content[:300])

Pages loaded: 1
Page 0 metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-21T19:05:41+00:00', 'title': 'Copy of Copy of Nikhilesh Ramoliya resume.pdf', 'moddate': '2026-02-21T19:05:41+00:00', 'keywords': 'DAHB-v6pUQY,BAE7fHbJwN0,0', 'author': 'Nik 25', 'source': 'resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}
---- page 0 content (first 300 chars) ----
Nikhilesh Ramoliya  
+ 9 1  -  8 4 6 9 1 7 5 2 9 9 n i k h i l e s h r a m o l i y a @ g m a i l . c o m L i n k e d I n
S K I L L S
L a n g u a g e s :  J a v a S c r i p t ,  T y p e S c r i p t ,  P y t h o n
F r o n t e n d :  R e a c t . j s ,  N e x t . j s ,  R e d u x ,  R e a c t  Q u e r y


### 1c. DirectoryLoader - many files at once

Load every matching file in a folder via a glob pattern. Good for bulk ingestion. Pass which `loader_cls` to use per file.

In [11]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_loader = DirectoryLoader(
    '.',                       # folder to scan
    glob='**/*.txt',           # all .txt, recursively
    loader_cls=TextLoader,     # loader used for each file
)
dir_docs = dir_loader.load()

print('Files loaded:', len(dir_docs))
for d in dir_docs:
    print(' -', d.metadata['source'])

Files loaded: 1
 - sample.txt


### 1d. WebBaseLoader - web pages

Fetch a URL, strip HTML, return text. Needs internet. (Uses `bs4` under the hood; install if missing: `uv add beautifulsoup4`.)

In [10]:
# Optional - needs internet + beautifulsoup4
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader('https://www.lanatussystems.com/')
web_docs = web_loader.load()

print('Docs:', len(web_docs))
print('Source:', web_docs[0].metadata.get('source'))
print('Content length:', len(web_docs[0].page_content))
# print(web_docs[0].page_content[:300].strip())
print(web_docs)

Docs: 1
Source: https://www.lanatussystems.com/
Content length: 4516
[Document(metadata={'source': 'https://www.lanatussystems.com/', 'title': 'Home | Lanatus Systems - Digital Innovation Leaders', 'description': 'Transform your business with cutting-edge technology solutions. We specialize in AI, cloud computing, and digital transformation.', 'language': 'en'}, page_content="Home | Lanatus Systems - Digital Innovation LeadersHomeServicesSuccess StoriesProductRetail Solutions↗Butterneck↗Supermarket Intelligence↗About UsContact UsLeading Digital Innovation Since 2021TransformYour BusinessDigitallyLanatus Systems — your trusted partner in AI-driven digital transformation. From custom software development to cloud solutions, we help businesses thrive in the digital age.Explore SolutionsOur ExpertiseTechnologyStack MasteryFrom cutting-edge frontend frameworks to robust backend solutions, we master the full spectrum of modern technologies.AIChatGPT, ClaudeFrontendReact, Next.jsBackendNode.j

**Other common loaders** (same `.load()` API):
- `CSVLoader` - one Document per row
- `JSONLoader` - extract fields via jq schema
- `UnstructuredMarkdownLoader` / `UnstructuredWordDocumentLoader` - md, docx
- `WikipediaLoader`, `ArxivLoader`, `YoutubeLoader` - online sources

Find them all: `from langchain_community.document_loaders import ...`

---
# 2. Transformers (Text Splitters)

**Why split?** Two reasons:
1. Embedding models have a max input size. A 50-page PDF won't fit in one vector.
2. Retrieval is better with small, focused chunks. You want to fetch *the relevant paragraph*, not the whole book.

A splitter takes `Document`s and returns more, smaller `Document`s. Two key knobs:
- **`chunk_size`** - max characters (or tokens) per chunk
- **`chunk_overlap`** - how many chars repeat between neighbor chunks (keeps context across the cut)

Splitters live in `langchain_text_splitters`.

### 2a. RecursiveCharacterTextSplitter (the default, use this)

Tries to split on `\n\n`, then `\n`, then ` `, then char - so it keeps paragraphs/sentences whole when possible. Best general-purpose splitter.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # small for demo so you see multiple chunks
    chunk_overlap=30,    # 30 chars shared between adjacent chunks
)

chunks = splitter.split_documents(docs)   # docs = the sample.txt from section 1a

print('Original docs:', len(docs))
print('After split   :', len(chunks))
print()
for i, c in enumerate(chunks):
    print(f'--- chunk {i} ({len(c.page_content)} chars) ---')
    print(c.page_content)
    print()

Notice metadata is **preserved** on every chunk, and the overlap means the end of one chunk repeats at the start of the next.

### 2b. CharacterTextSplitter - splits on one separator only

Simpler. Splits on a single separator (default `\n\n`). Less smart than recursive - a chunk can exceed `chunk_size` if no separator is found.

In [16]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(
    separator='\n',
    chunk_size=150,
    chunk_overlap=0,
)
char_chunks = char_splitter.split_documents(docs)
print('Chunks:', len(char_chunks))
print(char_chunks[4].page_content)

Chunks: 8
1. Load   - read raw data from files, web pages, databases, PDFs.
2. Split  - chop large documents into smaller chunks.


### 2c. Token-based splitting

Embedding/LLM limits are in **tokens**, not characters. Split by token count to respect those limits exactly. (`~4 chars = 1 token` rough rule.) Needs `tiktoken`: `uv add tiktoken`.

In [17]:
# Optional - needs tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=50,      # 50 TOKENS, not chars
    chunk_overlap=10,
)
token_chunks = token_splitter.split_documents(docs)
print('Token-based chunks:', len(token_chunks))
print(token_chunks[0].page_content)

Token-based chunks: 7
RAG stands for Retrieval Augmented Generation.


**Rules of thumb:**
- Default to `RecursiveCharacterTextSplitter`.
- `chunk_size` 500-1500 chars is common for prose; tune to your data.
- `chunk_overlap` ~10-20% of chunk_size to avoid cutting context.
- Code? Use `RecursiveCharacterTextSplitter.from_language(...)`.

---
# 3. Embeddings

An **embedding** turns text into a list of floats (a **vector**) that captures meaning. Similar meaning -> vectors close together in space. This is what makes semantic search possible: we embed the query and find the nearest chunk vectors.

Two methods on every embedding object:
- `embed_documents(list_of_texts)` -> list of vectors (for your chunks)
- `embed_query(single_text)` -> one vector (for the user's query)

We use Google's embedding model to match this project's stack.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2-preview')

vector = embeddings.embed_query('What is retrieval augmented generation?')

print('Vector dimension:', len(vector))
print('First 8 numbers:', vector[:8])

GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

### 3a. Embed all chunks

Take the chunks from section 2a and embed every one. This is what you'd store in a vector DB.

In [ ]:
texts = [c.page_content for c in chunks]
chunk_vectors = embeddings.embed_documents(texts)

print('Chunks embedded:', len(chunk_vectors))
print('Each vector dim :', len(chunk_vectors[0]))

### 3b. Semantic similarity - the core idea of retrieval

Cosine similarity measures how close two vectors are (1 = identical meaning, 0 = unrelated). Let's manually score a query against each chunk - this is exactly what a retriever does internally.

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

query = 'How does RAG reduce hallucination?'
q_vec = embeddings.embed_query(query)

scored = [(cosine(q_vec, v), txt) for v, txt in zip(chunk_vectors, texts)]
scored.sort(reverse=True)

print('Query:', query)
print()
for score, txt in scored:
    print(f'{score:.3f}  {txt[:70]!r}')

The top-scored chunk is the most relevant to the query. In a real RAG app you'd feed that chunk + the query to the LLM to generate a grounded answer.

---
## Recap

1. **Loaders** read raw data -> `Document(page_content, metadata)`. Same `.load()` API for txt, pdf, web, csv...
2. **Splitters** chop docs into chunks. `RecursiveCharacterTextSplitter` is the default; tune `chunk_size` + `chunk_overlap`.
3. **Embeddings** turn text -> vectors. `embed_documents` for chunks, `embed_query` for queries. Similar meaning = close vectors.

**Next steps:** store vectors in a vector DB (Chroma, FAISS), wrap it as a retriever, then chain retriever + LLM into a full RAG Q&A.